In [51]:
import pandas as pd, json, re, glob, os
import numpy as np
import os
from math import comb
from collections import defaultdict

CHUNK_NAME  = "chunk13"
COMPRESSION = 0.6     # 압축률 0.5 / 0.7 / 0.9
PARTIAL_AS  = 1       # 1=Any-token(KEEP·PARTIAL 생존), 0=Strict(KEEP만 생존)
ERROR_CATS  = {"VAPI"} 

# 원인 줄의 일부 토큰만 남아도 llm이 원인을 유추할수 있다고 보기때문에, partial도 1로 치환.(any-token)
# 다만 어떤 토큰이 남았는지의 기준으로 구분을 다 하지는 못한다.
# 그래서 strict(전부 남아야 생존)기준도 같이 돌려서 두 기준 다 확인한다.
# 0 | 1 로 나누는 이유는 멕네마 검정이 이진 비교법이기 때문에.

In [52]:
try:
    os.chdir(os.path.dirname(os.path.abspath(__file__)))
except NameError:
    pass

#실행위치를 파일에 있는 폴더로 맞추는 명령어

In [53]:
# 확장자를 가진 파일 모으기
def _find(kw, ext):
    h = [f for f in glob.glob(f"*{ext}") if kw in f.lower()] # 특정키워드 찾기.(kw)
    return h[0] if h else None #첫번째 파일만 반환.


# txt를 파싱. (베이스라인이 어떻게 결과를 냈나?)
#원본 결과를 인식하기 쉽게 정리. 예시 ->
# blocks = {
#     0.3: {1: "KEEP", 2: "DROP", 3: "PARTIAL"},
#     0.7: {1: "KEEP", 2: "DROP", 3: "DROP"},
# }
def _parse(path):
    data = open(path, encoding="utf-8", errors="replace").read() 
    blocks, cur = {}, None #결과 데이터, 현재 압축률
    for l in data.splitlines(): 
        m = re.match(r"압축률:\s*([\d.]+)", l) #형식에 맞게 매칭
        if m and "Line" not in l:
            cur = float(m.group(1)); blocks.setdefault(cur, {}) #setdefault: 없으면 빈 딕셔너리 파기
        lm = re.match(r"Line (\d+)\s*\|\s*(KEEP|PARTIAL|DROP|NO_TOKEN)", l)
        if lm and cur is not None:
            blocks[cur][int(lm.group(1))] = lm.group(2)
    return blocks
 
# 형식에 맞는 패턴을 찾아서, group(1)로 순수 텍스트 본문 꺼내기, 60글자만 잘라서 저장.
# 원본 파일(chunk.txt)과 비교. 0번라인이 원본의 i번째줄 시작점 리턴.
# 결과파일에 적힌 Line이 원본 파일 전체 기준으로 시작지점 찾는것. 1:1로 매핑
# --> 실제 압축률을 원본과 비교함. --> 줄어든 비율(drop)
def _offset(result_file, chunk_txt):
    data = open(result_file, encoding="utf-8", errors="replace").read()
    first = re.search(r"Line 00\s*\|\s*\w+\s*\|\s*\d+/\d+\s*\|\s*(.+)", data).group(1)[:60]
    for i, line in enumerate(open(chunk_txt, encoding="utf-8", errors="replace").read().splitlines()):
        if line[:60] == first:
            return i
    return None

In [54]:
# ── 실행: 폴더에서 자동으로 다 찾아서 변수 생성 ──
# csv정리 -> 진짜 정답 체크
_base_dir = "/Users/mac/무제 폴더/swm-logpress/part_D(select,review)/week7_logpress_evaluation/chunk_library"

_meta  = f"{_base_dir}/window_314927_metadata.csv"
_chunk = f"{_base_dir}/window_314927_chunk13.txt"
_r = pd.read_csv(_meta)

_r = _r[_r["chunk"] == CHUNK_NAME].iloc[0] #청크네임 불러들임
_start = _r["start_line"] # 해당 청크 시작지점
_alerts = json.loads(_r["alerts"])                       # {절대줄: 카테고리}
_cat = {int(k) - _start: v for k, v in _alerts.items()}  # chunk 내부줄: 카테고리
cause = sorted(_cat.keys())   # 청크가 발생한 내부 줄번호

In [55]:
_base_dir = "/Users/mac/무제 폴더/swm-logpress/part_D(select,review)/week7_logpress_evaluation/chunk_library"

_files = [f for f in glob.glob(f"{_base_dir}/*subchunk*.txt") # 서브청크 파일 다 합치기
          if CHUNK_NAME not in f and "metadata" not in f.lower()]
_method_status = defaultdict(dict)   # {방식: {chunk내부줄: status}}
for f in _files:
    name = re.split(r"_subchunk", os.path.basename(f))[0] # 베이스라인 이름 추출
    off = _offset(f, _chunk) 
    if off is None:
        print(f"[!] {f}: 위치 매칭 실패, 건너뜀"); continue
    blk = _parse(f).get(COMPRESSION, {})
    for local, st in blk.items(): #특정압축률만 꺼내오기
        _method_status[name][off + local] = st   # 로컬줄 → chunk13 전체줄
 
def _surv(status_map):
    return [1 if status_map.get(c) == "KEEP"
            else PARTIAL_AS if status_map.get(c) == "PARTIAL"
            else 0 for c in cause]
 
_methods = {n: _surv(s) for n, s in _method_status.items()} # 채점 결과 출력

In [56]:
# ── logpress / baseline 분리 ──
_lp = next((n for n in _methods if "logpress" in n.lower()), None)
logpress = _methods.get(_lp, [0] * len(cause))
baseline = {n: v for n, v in _methods.items() if n != _lp}

In [57]:
# ── is_error / pos / strata ──
# 앞선 채점 점수들을 조건부 세부성적 출력
is_error = {c: (_cat[c] in ERROR_CATS) for c in cause} #에러 카테고리목록 검사
_n = len(cause)
pos = {c: (cause.index(c) / (_n - 1) if _n > 1 else 0.0) for c in cause} #상대적 위치계산
 
overall   = list(range(_n)) #전체
non_error = [i for i in range(_n) if not is_error[cause[i]]] 
#에러가 아닌줄. is_cause가 false인줄. // 에러 키워드만 맹목적으로 살리는가? 중요한 일반, 경고 로그들도 다 버리는가?
middle    = [i for i in range(_n) if 0.25 <= pos[cause[i]] <= 0.75]
# 중간 위치줄, lostinthemiddle 평가
strata = {"overall": overall, "non_error": non_error, "middle": middle}
 
print(f"[준비 완료] 압축률 {COMPRESSION} · PARTIAL={'Any' if PARTIAL_AS else 'Strict'}")
print(f"  chunk13 alert(원인) {len(cause)}개 · 방식: {list(_methods.keys())}")
if _lp is None:
    print("  ※ LogPress 결과 없음 → logpress는 전부 0 (baseline끼리만 의미 있음)")
for n, v in {"logpress": logpress, **baseline}.items():
    print(f"  {n:20s}: {sum(v)}/{len(v)} 보존")
print(f"  strata: overall={len(overall)} non_error={len(non_error)} middle={len(middle)}")

[준비 완료] 압축률 0.6 · PARTIAL=Any
  chunk13 alert(원인) 54개 · 방식: []
  ※ LogPress 결과 없음 → logpress는 전부 0 (baseline끼리만 의미 있음)
  logpress            : 0/54 보존
  strata: overall=54 non_error=0 middle=26


In [58]:
# 보존율 ==(원인 줄 중 살아남은 것 수) / (전체 원인 줄 수) == 남아있는 비율(keep)
def preservation(result, target):
    if len(target) == 0:
        return 0.0
    return sum(1 for c in target if result[c]) / len(target)

print("-- 층별 원인 줄 개수 (보존율의 분모) --")
for s_name, target in strata.items():
    n = len(target)
    note = "  ← 비어있음: chunk13기준이기 때문. vapi는 T or F" if n == 0 else ""
    print(f"{s_name:10} {n:>4}개{note}")
print()

COL = 11

print("-- 층별 보존율 (baseline) --")
header = f"{'method':<10}" + "".join(f"{s:>{COL}}" for s in strata)
print(header)

for name, res in baseline.items():
    cells = ""
    for target in strata.values():
        if len(target) == 0:
            cells += f"{'-':>{COL}}"
        else:
            cells += f"{preservation(res, target):>{COL}.1%}"
    print(f"{name:<10}{cells}")

-- 층별 원인 줄 개수 (보존율의 분모) --
overall      54개
non_error     0개  ← 비어있음: chunk13기준이기 때문. vapi는 T or F
middle       26개

-- 층별 보존율 (baseline) --
method        overall  non_error     middle
